# Step 1 — 篩選文章
從 `stock_text` 撈出與目標公司相關的主文，輸出 `articles_raw.csv`

Big Data & Business Analytics, National Taiwan University

In [7]:
# ── 可調整參數區 ──────────────────────────────────────
COMPANY_ID   = '1519'    # 股票代號
COMPANY_NAME = '華城'    # 用於關鍵字搜尋
# ──────────────────────────────────────────────────────

import os
from dotenv import load_dotenv
load_dotenv()  # 讀取同目錄下的 .env 檔案

DB_CONFIG = {
    'host':     os.getenv('DB_HOST',    'localhost'),
    'user':     os.getenv('DB_USER',    'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME',    'bda2026'),
    'charset':  os.getenv('DB_CHARSET', 'utf8mb4')
}

In [2]:
import pymysql
import csv

In [3]:
# 連線資料庫，取得 stock_prices 的日期範圍
conn = pymysql.connect(**DB_CONFIG)
cursor = conn.cursor()

cursor.execute("""
    SELECT MIN(trade_date), MAX(trade_date)
    FROM stock_prices
    WHERE company_id = %s
""", (COMPANY_ID,))
min_date, max_date = cursor.fetchone()
print(f'股價資料範圍：{min_date} ～ {max_date}')

股價資料範圍：2023-03-01 ～ 2025-02-27


In [4]:
# [METHOD] 目前方法：LIKE 關鍵字比對 | 可替換為：Aho-Corasick 多模式掃描

# 篩選條件：
#   1. title 或 content 含公司名稱或股票代號
#   2. 只取主文（content_type = 'main' 或 IS NULL）
#   3. 文章時間在 stock_prices 有資料的期間內
cursor.execute("""
    SELECT no, post_time, title, content, s_name
    FROM stock_text
    WHERE (title   LIKE %s
        OR content LIKE %s
        OR title   LIKE %s
        OR content LIKE %s)
      AND (content_type = 'main' OR content_type IS NULL)
      AND DATE(post_time) >= %s
      AND DATE(post_time) <= %s
    ORDER BY post_time
""", (
    f'%{COMPANY_NAME}%', f'%{COMPANY_NAME}%',
    f'%{COMPANY_ID}%',   f'%{COMPANY_ID}%',
    min_date, max_date
))

rows = cursor.fetchall()
print(f'找到 {len(rows)} 篇相關文章')

cursor.close()
conn.close()

找到 4364 篇相關文章


In [5]:
# 儲存為 CSV
output_file = 'articles_raw.csv'

f = open(output_file, 'w', newline='', encoding='utf-8')
writer = csv.writer(f)
writer.writerow(['no', 'post_time', 'title', 'content', 's_name'])  # 欄位名稱
for row in rows:
    writer.writerow(row)
f.close()

print(f'已儲存至 {output_file}，共 {len(rows)} 筆')

已儲存至 articles_raw.csv，共 4364 筆


In [6]:
# 各來源平台分布
from collections import Counter
src_count = Counter(row[4] for row in rows)
for src, cnt in src_count.most_common():
    print(f'  {src}: {cnt} 篇')

  Yahoo股市: 2770 篇
  Ptt: 890 篇
  Yahoo新聞: 405 篇
  Dcard: 292 篇
  Mobile01: 7 篇
